<div dir="rtl" align="right">

# SVM با کرنل

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## ایده در یک نگاه
بعضی داده‌ها با خط راست جدا نمی‌شوند (مثلاً دایره‌ای درون دایره دیگر). ایده: داده را به **فضای بالاتر** ببریم که در آن جدا شود. مثلاً با ویژگی جدید $z=x_1^2+x_2^2$ (فاصله از مرکز)، دایره داخلی و بیرونی با یک صفحه ساده $z=c$ جدا می‌شوند.

## ترفند کرنل
SVM در فرم دوگان فقط به **ضرب داخلی** نقاط نیاز دارد: $\mathbf x_i^\top\mathbf x_j$. اگر آن را با یک **کرنل** $K(\mathbf x_i,\mathbf x_j)$ عوض کنیم، مثل این است که در یک فضای خیلی بزرگ (حتی بی‌نهایت‌بعدی) کار کنیم، **بدون اینکه آن فضا را بسازیم**.

$$f(\mathbf x)=\sum_{i\in SV}\alpha_iy_iK(\mathbf x_i,\mathbf x)+b$$

| کرنل | فرمول | پارامتر |
|---|---|---|
| خطی | $\mathbf x^\top\mathbf x'$ | — |
| چندجمله‌ای | $(\gamma\,\mathbf x^\top\mathbf x'+r)^d$ | `degree`، `coef0` |
| RBF (گاوسی) | $\exp(-\gamma\Vert\mathbf x-\mathbf x'\Vert^2)$ | `gamma` |
| سیگموید | $\tanh(\gamma\,\mathbf x^\top\mathbf x'+r)$ | کمتر استفاده می‌شود |

> [سه کرنل روی داده دایره‌ای: خطی شکست می‌خورد؛ چندجمله‌ای و RBF موفق‌اند.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/svm_kernels.png)

*سه کرنل روی داده دایره‌ای: خطی شکست می‌خورد؛ چندجمله‌ای و RBF موفق‌اند.*



</div>

In [ ]:
import numpy as np
from sklearn.datasets import make_circles
from sklearn.svm import SVC

X, y = make_circles(300, noise=0.08, factor=0.5, random_state=0)
print("linear on x1, x2       :", round(SVC(kernel="linear").fit(X, y).score(X, y), 3))
# Add the feature z = x1^2 + x2^2 by hand: now a linear SVM works
Z = np.c_[X, (X ** 2).sum(1)]
print("linear on x1, x2, z    :", round(SVC(kernel="linear").fit(Z, y).score(Z, y), 3))
print("RBF kernel on x1, x2   :", round(SVC(kernel="rbf").fit(X, y).score(X, y), 3))

# The RBF kernel value between two points
a, b = np.array([0, 0]), np.array([1, 1])
for g in [0.1, 1, 10]:
    print(f"K(a, b) with gamma={g}: {np.exp(-g * np.sum((a - b) ** 2)):.4f}")

<div dir="rtl" align="right">



## gamma و C با هم
- **gamma:** شعاع اثر هر بردار پشتیبان. کوچک = اثر گسترده و مرز صاف؛ بزرگ = اثر محلی و مرز پرپیچ‌وخم.
- **C:** چقدر برای خطای آموزش جریمه بدهیم.

[[lab:cls_ksvm]]

در آزمایشگاه، gamma را خیلی بزرگ کن: دور هر نقطه یک «جزیره» شکل می‌گیرد (بیش‌برازش).

| | gamma کوچک | gamma بزرگ |
|---|---|---|
| C کوچک | خیلی ساده | محلی ولی نرم |
| C بزرگ | مرز صاف و دقیق | بیش‌برازش شدید |

> همیشه gamma و C را **با هم** روی شبکه لگاریتمی جستجو کن، مثلاً $C\in\{0.1,1,10,100\}$ و $\gamma\in\{0.001,0.01,0.1,1\}$.



</div>

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_breast_cancer(return_X_y=True)
grid = GridSearchCV(make_pipeline(StandardScaler(), SVC()), {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1, 1]}, cv=5)
grid.fit(X, y)
print("Best:", grid.best_params_, round(grid.best_score_, 3))
table = pd.DataFrame(grid.cv_results_)[["param_svc__C", "param_svc__gamma", "mean_test_score"]]
print(table.pivot(index="param_svc__C", columns="param_svc__gamma", values="mean_test_score").round(3))

<div dir="rtl" align="right">



| مزایا | معایب |
|---|---|
| مرزهای غیرخطی دقیق | کند روی بیش از ده‌ها هزار نمونه |
| با ابعاد بالا (متن) خوب کار می‌کند | انتخاب کرنل و پارامترها |
| حافظه کم: فقط بردارهای پشتیبان | تفسیر سخت |

</div>

<div dir="rtl" align="right">

### تمرین

روی داده دو هلال (make_moons) کرنل‌های linear، poly و rbf را مقایسه کن.

</div>

In [ ]:
# Your answer here